In [3]:
import pandas as pd
import numpy as np

# =========================
# USER INPUTS
# =========================
AMAZON_FEE_PCT = 0.35
COGS_PER_PAIR = 2.59

LOAN_AMOUNT = 12000
LOAN_TOTAL_PAYBACK = 13584
LOAN_TERM_MONTHS = 24
LOAN_PAYMENT_MONTHLY = LOAN_TOTAL_PAYBACK / LOAN_TERM_MONTHS

CREDIT_CARD_PAYMENT_MONTHLY = 2000   # <-- UPDATED
OTHER_FIXED_EXPENSES_MONTHLY = 0.0
BEGINNING_CASH = 0.0

FORECAST_MONTHS = 6
INVENTORY_PURCHASE_MONTH_1 = 4500


# =========================
# DAILY MODEL (UPDATED)
# =========================
def build_daily_model(sales_df: pd.DataFrame) -> pd.DataFrame:
    df = sales_df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    daily = (
        df.groupby("date", as_index=False)
        .agg(
            revenue=("revenue", "sum"),
            pairs_sold=("pairs_sold", "sum"),
            orders=("amazon-order-id", "nunique"),
        )
        .sort_values("date")
    )

    daily["amazon_fees"] = daily["revenue"] * AMAZON_FEE_PCT
    daily["cogs"] = daily["pairs_sold"] * COGS_PER_PAIR

    # IMPORTANT: PPC REMOVED
    daily["net_before_cc"] = (
        daily["revenue"] - daily["amazon_fees"] - daily["cogs"]
    )

    daily["month"] = daily["date"].dt.to_period("M").dt.to_timestamp()

    return daily


# =========================
# MONTHLY HISTORICAL
# =========================
def summarize_historical_monthly(daily_df: pd.DataFrame) -> pd.DataFrame:
    monthly = (
        daily_df.groupby("month", as_index=False)
        .agg(
            revenue=("revenue", "sum"),
            pairs_sold=("pairs_sold", "sum"),
            amazon_fees=("amazon_fees", "sum"),
            cogs=("cogs", "sum"),
            net_before_cc=("net_before_cc", "sum"),
        )
        .sort_values("month")
    )
    return monthly


# =========================
# SIMPLE FORECAST (REALISTIC)
# =========================
def simple_6m_forecast(
    historical_monthly: pd.DataFrame,
    scenario: str = "base",
    forecast_months: int = 6
) -> pd.DataFrame:

    df = historical_monthly.copy().sort_values("month")

    recent = df.tail(3)

    base_revenue = recent["revenue"].mean()
    base_pairs_ratio = recent["pairs_sold"].sum() / recent["revenue"].sum()

    scenario_mult = {
        "stress": 0.75,
        "conservative": 0.85,
        "base": 1.00,
        "upside": 1.10,
    }[scenario]

    last_month = df["month"].max()

    future_months = pd.date_range(
        last_month + pd.offsets.MonthBegin(1),
        periods=forecast_months,
        freq="MS"
    )

    out = pd.DataFrame({"month": future_months})

    out["revenue"] = base_revenue * scenario_mult
    out["pairs_sold"] = out["revenue"] * base_pairs_ratio
    out["amazon_fees"] = out["revenue"] * AMAZON_FEE_PCT
    out["cogs"] = out["pairs_sold"] * COGS_PER_PAIR

    # KEY METRIC
    out["net_before_cc"] = (
        out["revenue"] - out["amazon_fees"] - out["cogs"]
    )

    return out


# =========================
# CASHFLOW (FIXED)
# =========================
def add_cashflow(
    forecast_df: pd.DataFrame,
    loan_payment: float,
    credit_card_payment: float,
    beginning_cash: float,
    loan_amount: float,
    inventory_purchase: float,
) -> pd.DataFrame:

    df = forecast_df.copy().sort_values("month").reset_index(drop=True)

    df["loan_payment"] = loan_payment
    df["credit_card_payment"] = credit_card_payment

    df["net_after_debt"] = (
        df["net_before_cc"]
        - df["loan_payment"]
        - df["credit_card_payment"]
    )

    cash = []
    running_cash = beginning_cash + loan_amount

    for i, row in df.iterrows():

        # inventory hit in first month
        if i == 0:
            running_cash -= inventory_purchase

        running_cash += row["net_after_debt"]
        cash.append(running_cash)

    df["ending_cash"] = cash

    return df


# =========================
# RUN MODEL
# =========================

daily_model = build_daily_model(sales_df)
historical_monthly = summarize_historical_monthly(daily_model)

forecast_base = simple_6m_forecast(historical_monthly, "base")
forecast_cons = simple_6m_forecast(historical_monthly, "conservative")
forecast_stress = simple_6m_forecast(historical_monthly, "stress")

cashflow_base = add_cashflow(
    forecast_base,
    LOAN_PAYMENT_MONTHLY,
    CREDIT_CARD_PAYMENT_MONTHLY,
    BEGINNING_CASH,
    LOAN_AMOUNT,
    INVENTORY_PURCHASE_MONTH_1
)

cashflow_cons = add_cashflow(
    forecast_cons,
    LOAN_PAYMENT_MONTHLY,
    CREDIT_CARD_PAYMENT_MONTHLY,
    BEGINNING_CASH,
    LOAN_AMOUNT,
    INVENTORY_PURCHASE_MONTH_1
)

cashflow_stress = add_cashflow(
    forecast_stress,
    LOAN_PAYMENT_MONTHLY,
    CREDIT_CARD_PAYMENT_MONTHLY,
    BEGINNING_CASH,
    LOAN_AMOUNT,
    INVENTORY_PURCHASE_MONTH_1
)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("\nHISTORICAL")
print(historical_monthly.round(2))

print("\nBASE CASE")
print(cashflow_base.round(2))

print("\nCONSERVATIVE CASE")
print(cashflow_cons.round(2))

print("\nSTRESS CASE")
print(cashflow_stress.round(2))


# =========================
# DECISION CHECK
# =========================
def decision_check(df, label):
    print(f"\n{label}")
    print(f"Min cash: {df['ending_cash'].min():,.2f}")
    print(f"Final cash: {df['ending_cash'].iloc[-1]:,.2f}")
    print(f"Months negative: {(df['ending_cash'] < 0).sum()}")


decision_check(cashflow_base, "BASE")
decision_check(cashflow_cons, "CONSERVATIVE")
decision_check(cashflow_stress, "STRESS")


HISTORICAL
        month  revenue  pairs_sold  amazon_fees     cogs  net_before_cc
0  2025-02-01   153.89          11        53.86    28.49          71.54
1  2025-03-01   657.53          47       230.14   121.73         305.66
2  2025-04-01   629.55          45       220.34   116.55         292.66
3  2025-05-01   754.45          57       264.06   147.63         342.76
4  2025-06-01 1,706.86         133       597.40   344.47         764.99
5  2025-07-01 2,424.11         198       848.44   512.82       1,062.85
6  2025-08-01 2,363.90         190       827.36   492.10       1,044.44
7  2025-09-01 1,735.09         136       607.28   352.24         775.57
8  2025-10-01 3,698.43         305     1,294.45   789.95       1,614.03
9  2025-11-01 3,742.90         349     1,310.01   903.91       1,528.98
10 2025-12-01 2,827.85         229       989.75   593.11       1,244.99
11 2026-01-01 2,296.10         178       803.64   461.02       1,031.45
12 2026-02-01 3,383.65         266     1,184.28   68

/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_32908/3897652183.py:198: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(historical_monthly.round(2))
/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_32908/3897652183.py:201: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(cashflow_base.round(2))
/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_32908/3897652183.py:204: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(cashflow_cons.round(2))
/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_32908/3897652183.py:207: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(cashflow_stress.round(2))
